In [11]:
%pip install scikit-learn scipy numpy pandas tqdm


Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
import numpy as np

from collections import defaultdict
from tqdm.auto import tqdm
from sklearn.cluster import MiniBatchKMeans

In [13]:
train_path = "train.csv"
dest_path = "destinations.csv"
test_path = "test.csv"

In [23]:
def add_to_dict(d, key, cluster, score):
    d[key][int(cluster)] += float(score)


def top5_from_counter(counter):
    return [k for k, _ in sorted(counter.items(), key=lambda x: (-x[1], x[0]))[:5]]


def build_top5_map(d):
    return {k: top5_from_counter(v) for k, v in d.items()}


def merge_preds(*parts):
    result = []
    for part in parts:
        if part is None:
            continue
        for x in part:
            x = int(x)
            if x not in result:
                result.append(x)
            if len(result) == 5:
                return result
    return result


def safe_int(x):
    if pd.isna(x):
        return None
    return int(x)


def safe_float(x, ndigits=3):
    if pd.isna(x):
        return None
    return round(float(x), ndigits)


In [24]:
best_od_ulc = defaultdict(lambda: defaultdict(float))
best_search_dest = defaultdict(lambda: defaultdict(float))
best_market = defaultdict(lambda: defaultdict(float))
best_country = defaultdict(lambda: defaultdict(float))
best_dest = defaultdict(lambda: defaultdict(float))
popular = defaultdict(float)


In [25]:
usecols = [
    "date_time",
    "user_location_country",
    "user_location_region",
    "user_location_city",
    "orig_destination_distance",
    "srch_destination_id",
    "hotel_country",
    "hotel_market",
    "is_booking",
    "cnt",
    "hotel_cluster",
]

dtypes = {
    "user_location_country": "float32",
    "user_location_region": "float32",
    "user_location_city": "float32",
    "orig_destination_distance": "float32",
    "srch_destination_id": "int32",
    "hotel_country": "float32",
    "hotel_market": "float32",
    "is_booking": "int8",
    "cnt": "int16",
    "hotel_cluster": "int8",
}

reader = pd.read_csv(train_path, usecols=usecols, dtype=dtypes, chunksize=500000)


In [26]:
for chunk in tqdm(reader):
    chunk["date_time"] = pd.to_datetime(chunk["date_time"], errors="coerce")
    chunk["year"] = chunk["date_time"].dt.year.fillna(2013).astype(int)
    chunk["month"] = chunk["date_time"].dt.month.fillna(1).astype(int)
    chunk["orig_destination_distance"] = chunk["orig_destination_distance"].round(3)

    for row in chunk.itertuples(index=False):
        year = int(row.year)
        month = int(row.month)
        ulc = safe_int(row.user_location_country)
        ulr = safe_int(row.user_location_region)
        ulc_city = safe_int(row.user_location_city)
        odd = safe_float(row.orig_destination_distance, 3)
        sdi = int(row.srch_destination_id)
        hc = safe_int(row.hotel_country)
        hm = safe_int(row.hotel_market)
        is_booking = int(row.is_booking)
        cnt = int(row.cnt)
        cluster = int(row.hotel_cluster)

        time_weight = (year - 2012) * 12 + month
        base_score = time_weight * (1.0 + 4.0 * is_booking) * np.log1p(cnt)

        if (
            ulc is not None
            and ulr is not None
            and ulc_city is not None
            and odd is not None
        ):
            add_to_dict(
                best_od_ulc, (ulc, ulr, ulc_city, odd), cluster, base_score * 1.2
            )

        if sdi is not None and hc is not None and hm is not None:
            add_to_dict(best_search_dest, (sdi, hc, hm), cluster, base_score * 1.1)

        if hm is not None:
            add_to_dict(best_market, (hm,), cluster, base_score)

        if hc is not None:
            add_to_dict(best_country, (hc,), cluster, base_score * 0.8)

        add_to_dict(best_dest, (sdi,), cluster, base_score * 0.9)
        popular[cluster] += base_score


0it [00:00, ?it/s]

In [27]:
top_od_ulc = build_top5_map(best_od_ulc)
top_search_dest = build_top5_map(best_search_dest)
top_market = build_top5_map(best_market)
top_country = build_top5_map(best_country)
top_dest = build_top5_map(best_dest)
top_popular = top5_from_counter(popular)

top_popular


[91, 48, 41, 64, 65]

In [28]:
test_df = pd.read_csv(test_path)
test_df["orig_destination_distance"] = test_df["orig_destination_distance"].round(3)
test_df.head()


,date_time,site_name,posa_continent,user_location_country,user_location_region,user_location_city,orig_destination_distance,user_id,is_mobile,is_package,...,srch_adults_cnt,srch_children_cnt,srch_rm_cnt,srch_destination_id,srch_destination_type_id,is_booking,cnt,hotel_continent,hotel_country,hotel_market
0,2013-08-25 16:56:20,2,3,66,174,46432,2800.886,144246,0,1,...,2,0,1,399,1,0,1,4,80,204
1,2014-05-10 15:39:28,2,3,66,337,25814,NaN,339108,0,0,...,1,0,1,12509,5,0,1,4,8,113
2,2013-07-09 07:22:59,2,3,69,1005,48189,NaN,340344,0,0,...,2,0,1,13022,4,0,3,6,70,10
3,2013-03-31 16:13:50,27,2,167,47,23978,NaN,220786,0,0,...,2,0,1,11815,1,0,1,3,5,44
4,2013-10-03 12:03:47,2,3,66,174,9351,6.272,347602,0,1,...,1,0,1,8279,1,0,1,2,50,1230


In [29]:
predictions = []

for row in tqdm(test_df.itertuples(index=False), total=len(test_df)):
    ulc = safe_int(getattr(row, "user_location_country"))
    ulr = safe_int(getattr(row, "user_location_region"))
    ulc_city = safe_int(getattr(row, "user_location_city"))
    odd = safe_float(getattr(row, "orig_destination_distance"), 3)
    sdi = safe_int(getattr(row, "srch_destination_id"))
    hc = safe_int(getattr(row, "hotel_country"))
    hm = safe_int(getattr(row, "hotel_market"))

    p1 = None
    if ulc is not None and ulr is not None and ulc_city is not None and odd is not None:
        p1 = top_od_ulc.get((ulc, ulr, ulc_city, odd))

    p2 = None
    if sdi is not None and hc is not None and hm is not None:
        p2 = top_search_dest.get((sdi, hc, hm))

    p3 = None
    if hm is not None:
        p3 = top_market.get((hm,))

    p4 = None
    if hc is not None:
        p4 = top_country.get((hc,))

    p5 = None
    if sdi is not None:
        p5 = top_dest.get((sdi,))

    pred = merge_preds(p1, p2, p5, p3, p4, top_popular)
    predictions.append(pred)


  0%|          | 0/10000 [00:00<?, ?it/s]

In [30]:
print(str(predictions).replace(" ", ""))


[[89,53,26,84,73],[82,28,36,12,85],[97,25,90,64,58],[63,92,58,62,57],[47,98,70,41,56],[26,73,84,92,0],[95,98,21,68,72],[1,45,79,24,54],[36,62,46,85,67],[46,36,62,58,9],[55,9,21,95,68],[71,34,77,0,54],[65,31,52,87,64],[62,29,12,36,85],[40,17,83,30,23],[64,46,99,25,11],[25,59,0,31,96],[68,9,91,28,59],[91,48,32,77,42],[65,52,87,66,80],[57,81,63,62,85],[82,29,11,5,46],[95,91,18,98,68],[34,28,22,63,32],[40,1,45,79,24],[2,25,97,98,64],[36,82,62,30,85],[53,73,63,90,96],[59,2,25,33,46],[91,72,0,28,21],[63,36,44,46,57],[91,48,42,28,99],[64,9,97,85,5],[96,71,34,77,0],[68,9,91,28,59],[44,92,71,34,77],[65,52,66,31,84],[91,32,50,15,42],[23,95,91,55,49],[2,36,62,59,82],[17,65,66,31,52],[36,46,81,67,53],[37,55,83,72,10],[64,97,8,46,11],[71,34,77,0,54],[72,25,59,16,47],[21,56,70,41,83],[78,61,81,14,7],[82,99,62,15,28],[91,18,59,39,28],[40,39,7,94,42],[83,91,42,18,48],[25,95,19,4,18],[46,36,29,12,62],[71,34,77,0,54],[67,62,58,36,64],[13,72,48,83,50],[98,83,73,25,17],[62,36,67,82,85],[33,28,83,91,34],[6